In [2]:
!pip install PyPDF2 python-docx

In [1]:
!ollama pull llama3.1

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [2]:
#!/usr/bin/env python3
"""
Enhanced Document-based Q&A Chatbot using Llama 3.1 via Ollama
Improved accuracy with semantic search, better chunking, and enhanced prompting
"""

import os
import requests
import json
import re
from pathlib import Path
import argparse
from typing import List, Dict, Optional, Tuple
from collections import Counter
import math

# Document processing imports
try:
    import PyPDF2
    from docx import Document
    PDF_SUPPORT = True
    DOCX_SUPPORT = True
except ImportError:
    PDF_SUPPORT = False
    DOCX_SUPPORT = False
    print("Install PyPDF2 and python-docx for full document support:")
    print("pip install PyPDF2 python-docx")

# Optional: Enhanced text processing
try:
    import nltk
    from nltk.tokenize import sent_tokenize, word_tokenize
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer
    NLTK_SUPPORT = True
    
    # Download required NLTK data
    try:
        nltk.data.find('tokenizers/punkt')
        nltk.data.find('corpora/stopwords')
    except LookupError:
        print("Downloading NLTK data...")
        nltk.download('punkt', quiet=True)
        nltk.download('stopwords', quiet=True)
        
except ImportError:
    NLTK_SUPPORT = False
    print("For better text processing, install NLTK: pip install nltk")

class EnhancedDocumentProcessor:
    """Enhanced document processing with better chunking and preprocessing"""
    
    def __init__(self):
        if NLTK_SUPPORT:
            self.stemmer = PorterStemmer()
            self.stop_words = set(stopwords.words('english'))
        else:
            self.stemmer = None
            self.stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should'}
    
    def load_document(self, file_path: str) -> str:
        """Load and extract text from various document formats"""
        path = Path(file_path)
        
        if not path.exists():
            raise FileNotFoundError(f"Document not found: {file_path}")
        
        if path.suffix.lower() == '.txt':
            return self._load_txt(file_path)
        elif path.suffix.lower() == '.pdf' and PDF_SUPPORT:
            return self._load_pdf(file_path)
        elif path.suffix.lower() == '.docx' and DOCX_SUPPORT:
            return self._load_docx(file_path)
        else:
            raise ValueError(f"Unsupported file format: {path.suffix}")
    
    def _load_txt(self, file_path: str) -> str:
        """Load text file"""
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    
    def _load_pdf(self, file_path: str) -> str:
        """Extract text from PDF"""
        text = ""
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
        return text
    
    def _load_docx(self, file_path: str) -> str:
        """Extract text from DOCX"""
        doc = Document(file_path)
        text = ""
        for paragraph in doc.paragraphs:
            text += paragraph.text + "\n"
        return text
    
    def preprocess_text(self, text: str) -> str:
        """Clean and preprocess text"""
        # Remove extra whitespace and normalize
        text = re.sub(r'\s+', ' ', text.strip())
        
        # Remove page numbers and headers/footers (simple heuristic)
        lines = text.split('\n')
        cleaned_lines = []
        
        for line in lines:
            line = line.strip()
            # Skip very short lines that might be page numbers or headers
            if len(line) < 10 and line.isdigit():
                continue
            # Skip lines with only special characters
            if re.match(r'^[^\w]*$', line):
                continue
            cleaned_lines.append(line)
        
        return '\n'.join(cleaned_lines)
    
    def smart_chunk_text(self, text: str, chunk_size: int = 800, overlap: int = 150) -> List[Dict]:
        """Enhanced chunking with semantic boundaries and metadata"""
        text = self.preprocess_text(text)
        
        if len(text) <= chunk_size:
            return [{"text": text, "start": 0, "end": len(text), "chunk_id": 0}]
        
        chunks = []
        chunk_id = 0
        
        # Try to use NLTK for sentence tokenization
        if NLTK_SUPPORT:
            sentences = sent_tokenize(text)
        else:
            # Fallback: simple sentence splitting
            sentences = re.split(r'[.!?]+', text)
            sentences = [s.strip() for s in sentences if s.strip()]
        
        current_chunk = ""
        current_start = 0
        
        for i, sentence in enumerate(sentences):
            # Calculate potential new chunk size
            potential_chunk = current_chunk + " " + sentence if current_chunk else sentence
            
            if len(potential_chunk) <= chunk_size:
                current_chunk = potential_chunk
            else:
                # Save current chunk if it has content
                if current_chunk:
                    chunk_end = current_start + len(current_chunk)
                    chunks.append({
                        "text": current_chunk.strip(),
                        "start": current_start,
                        "end": chunk_end,
                        "chunk_id": chunk_id,
                        "sentence_count": len(re.split(r'[.!?]+', current_chunk))
                    })
                    chunk_id += 1
                
                # Start new chunk with overlap
                if len(sentence) > chunk_size:
                    # Handle very long sentences
                    words = sentence.split()
                    for j in range(0, len(words), chunk_size // 10):
                        word_chunk = " ".join(words[j:j + chunk_size // 10])
                        chunks.append({
                            "text": word_chunk,
                            "start": text.find(word_chunk),
                            "end": text.find(word_chunk) + len(word_chunk),
                            "chunk_id": chunk_id,
                            "sentence_count": 1
                        })
                        chunk_id += 1
                else:
                    current_chunk = sentence
                    current_start = text.find(sentence, current_start)
        
        # Add final chunk
        if current_chunk:
            chunk_end = current_start + len(current_chunk)
            chunks.append({
                "text": current_chunk.strip(),
                "start": current_start,
                "end": chunk_end,
                "chunk_id": chunk_id,
                "sentence_count": len(re.split(r'[.!?]+', current_chunk))
            })
        
        return chunks

class SemanticSearcher:
    """Enhanced search using TF-IDF and semantic similarity"""
    
    def __init__(self):
        if NLTK_SUPPORT:
            self.stemmer = PorterStemmer()
            self.stop_words = set(stopwords.words('english'))
        else:
            self.stemmer = None
            self.stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should'}
    
    def preprocess_query(self, text: str) -> List[str]:
        """Preprocess text for semantic search"""
        # Convert to lowercase and tokenize
        if NLTK_SUPPORT:
            words = word_tokenize(text.lower())
        else:
            words = re.findall(r'\b\w+\b', text.lower())
        
        # Remove stopwords and stem
        processed_words = []
        for word in words:
            if word not in self.stop_words and len(word) > 2:
                if self.stemmer:
                    word = self.stemmer.stem(word)
                processed_words.append(word)
        
        return processed_words
    
    def calculate_tfidf_scores(self, query_words: List[str], chunks: List[Dict]) -> List[Tuple[float, Dict]]:
        """Calculate TF-IDF scores for relevance ranking"""
        # Calculate term frequencies in chunks
        chunk_word_counts = []
        all_words = set()
        
        for chunk in chunks:
            words = self.preprocess_query(chunk["text"])
            word_count = Counter(words)
            chunk_word_counts.append(word_count)
            all_words.update(words)
        
        # Calculate IDF (Inverse Document Frequency)
        idf = {}
        total_chunks = len(chunks)
        
        for word in all_words:
            chunks_with_word = sum(1 for wc in chunk_word_counts if word in wc)
            idf[word] = math.log(total_chunks / (chunks_with_word + 1))
        
        # Calculate TF-IDF scores for each chunk
        scores = []
        for i, chunk in enumerate(chunks):
            score = 0
            word_count = chunk_word_counts[i]
            total_words = sum(word_count.values())
            
            for query_word in query_words:
                if query_word in word_count:
                    tf = word_count[query_word] / total_words
                    score += tf * idf.get(query_word, 0)
            
            # Boost score based on chunk metadata
            if chunk.get("sentence_count", 1) > 3:  # Prefer chunks with more sentences
                score *= 1.1
            
            scores.append((score, chunk))
        
        return scores
    
    def find_relevant_chunks(self, question: str, chunks: List[Dict], max_chunks: int = 3) -> List[str]:
        """Find most relevant chunks using enhanced semantic search"""
        if not chunks:
            return []
        
        query_words = self.preprocess_query(question)
        if not query_words:
            return [chunk["text"] for chunk in chunks[:max_chunks]]
        
        # Calculate TF-IDF scores
        scored_chunks = self.calculate_tfidf_scores(query_words, chunks)
        
        # Sort by relevance score
        scored_chunks.sort(reverse=True, key=lambda x: x[0])
        
        # Filter out chunks with zero relevance
        relevant_chunks = [chunk for score, chunk in scored_chunks if score > 0]
        
        # If no relevant chunks found, fall back to first few chunks
        if not relevant_chunks:
            relevant_chunks = chunks[:max_chunks]
        else:
            relevant_chunks = relevant_chunks[:max_chunks]
        
        return [chunk["text"] if isinstance(chunk, dict) else chunk for chunk in relevant_chunks]

class EnhancedOllamaClient:
    """Enhanced Ollama client with improved prompting"""
    
    def __init__(self, base_url: str = "http://localhost:11434", model: str = "llama3.1"):
        self.base_url = base_url
        self.model = model
    
    def is_available(self) -> bool:
        """Check if Ollama is running"""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=5)
            return response.status_code == 200
        except:
            return False
    
    def check_model(self) -> bool:
        """Check if the specified model is available"""
        try:
            response = requests.get(f"{self.base_url}/api/tags")
            models = response.json().get('models', [])
            return any(self.model in model['name'] for model in models)
        except:
            return False
    
    def generate_response(self, prompt: str, context: str = "", question_type: str = "general") -> str:
        """Generate response with enhanced prompting"""
        full_prompt = self._build_enhanced_prompt(prompt, context, question_type)
        
        # Adjust parameters based on question type
        temperature = 0.3 if question_type == "factual" else 0.7
        
        payload = {
            "model": self.model,
            "prompt": full_prompt,
            "stream": False,
            "options": {
                "temperature": temperature,
                "top_p": 0.9,
                "max_tokens": 800,
                "stop": ["Human:", "Assistant:", "Context:"]
            }
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/api/generate", 
                json=payload,
                timeout=45
            )
            response.raise_for_status()
            return response.json()['response'].strip()
        except requests.RequestException as e:
            return f"Error communicating with Ollama: {str(e)}"
    
    def _classify_question(self, question: str) -> str:
        """Classify question type for better prompting"""
        question_lower = question.lower()
        
        if any(word in question_lower for word in ['what', 'who', 'when', 'where', 'which', 'how many']):
            return "factual"
        elif any(word in question_lower for word in ['why', 'how', 'explain', 'describe']):
            return "explanatory"
        elif any(word in question_lower for word in ['should', 'recommend', 'suggest', 'best']):
            return "advisory"
        else:
            return "general"
    
    def _build_enhanced_prompt(self, question: str, context: str, question_type: str = None) -> str:
        """Build enhanced prompt with better instructions"""
        if not question_type:
            question_type = self._classify_question(question)
        
        if context:
            return f"""You are an expert document analyst. Your task is to answer questions accurately based ONLY on the provided document context.

INSTRUCTIONS:
1. Read the context carefully and thoroughly
2. Answer based ONLY on information explicitly stated in the context
3. If the information is not in the context, clearly state "The document does not contain information about [specific aspect]"
4. Be precise and cite specific details from the context
5. If the context provides partial information, acknowledge what is known and what is missing
6. For {question_type} questions, focus on providing {self._get_question_guidance(question_type)}

DOCUMENT CONTEXT:
{context}

QUESTION: {question}

ANALYSIS AND ANSWER:
Based on the provided document context above, here is my response:"""
        else:
            return f"""I need document context to answer your question: {question}
Please provide relevant document content for me to analyze."""
    
    def _get_question_guidance(self, question_type: str) -> str:
        """Get specific guidance based on question type"""
        guidance = {
            "factual": "specific facts, numbers, dates, and concrete details",
            "explanatory": "clear explanations with supporting details from the document", 
            "advisory": "recommendations based on the information provided in the document",
            "general": "comprehensive information from the relevant sections"
        }
        return guidance.get(question_type, "accurate information from the document")

class EnhancedDocumentChatbot:
    """Enhanced chatbot with improved accuracy"""
    
    def __init__(self, model: str = "llama3.1"):
        self.processor = EnhancedDocumentProcessor()
        self.searcher = SemanticSearcher()
        self.ollama = EnhancedOllamaClient(model=model)
        self.document_chunks = []
        self.document_name = ""
        self.original_text = ""
    
    def load_document(self, file_path: str) -> bool:
        """Load and process document with enhanced processing"""
        print(f"Loading document: {file_path}")
        
        try:
            self.original_text = self.processor.load_document(file_path)
            self.document_chunks = self.processor.smart_chunk_text(self.original_text)
            self.document_name = Path(file_path).name
            
            print(f"✓ Document loaded successfully!")
            print(f"  - File: {self.document_name}")
            print(f"  - Text length: {len(self.original_text)} characters")
            print(f"  - Smart chunks created: {len(self.document_chunks)}")
            
            # Show sample chunk info
            if self.document_chunks:
                avg_chunk_size = sum(len(chunk["text"]) for chunk in self.document_chunks) / len(self.document_chunks)
                print(f"  - Average chunk size: {avg_chunk_size:.0f} characters")
            
        except Exception as e:
            print(f"✗ Error loading document: {str(e)}")
            return False
        
        return True
    
    def answer_question(self, question: str, max_chunks: int = 3, debug: bool = False) -> str:
        """Answer question with enhanced accuracy"""
        if not self.document_chunks:
            return "No document loaded. Please upload a document first."
        
        # Find relevant context using enhanced search
        relevant_chunks = self.searcher.find_relevant_chunks(question, self.document_chunks, max_chunks)
        
        if debug:
            print(f"Debug: Found {len(relevant_chunks)} relevant chunks")
            for i, chunk in enumerate(relevant_chunks):
                print(f"Chunk {i+1}: {chunk[:100]}...")
        
        context = "\n\n".join(relevant_chunks)
        
        # Classify question type for better prompting
        question_type = self.ollama._classify_question(question)
        
        if debug:
            print(f"Debug: Question classified as '{question_type}'")
        
        # Generate answer with enhanced prompting
        answer = self.ollama.generate_response(question, context, question_type)
        return answer
    
    def get_document_summary(self) -> str:
        """Generate a summary of the loaded document"""
        if not self.document_chunks:
            return "No document loaded."
        
        # Use first few chunks for summary
        summary_context = "\n".join([chunk["text"] for chunk in self.document_chunks[:3]])
        summary = self.ollama.generate_response(
            "Provide a brief summary of this document, highlighting the main topics and key points.",
            summary_context,
            "explanatory"
        )
        return summary
    
    def chat_loop(self):
        """Enhanced interactive chat loop"""
        print("\n" + "="*70)
        print("🚀 Enhanced Document Q&A Chatbot with Llama 3.1")
        print("="*70)
        
        # Check Ollama availability
        if not self.ollama.is_available():
            print("✗ Ollama is not running. Please start Ollama first:")
            print("  ollama serve")
            return
        
        if not self.ollama.check_model():
            print(f"✗ Model '{self.ollama.model}' not found. Please pull it first:")
            print(f"  ollama pull {self.ollama.model}")
            return
        
        print(f"✓ Connected to Ollama (Model: {self.ollama.model})")
        
        # Load document
        while not self.document_chunks:
            doc_path = input("\nEnter path to your document (PDF/TXT/DOCX): ").strip()
            if not doc_path:
                print("Goodbye!")
                return
            
            if not self.load_document(doc_path):
                continue
        
        # Offer document summary
        print(f"\nDocument '{self.document_name}' is ready!")
        if input("Would you like a document summary? (y/n): ").lower().startswith('y'):
            print("\n📄 Document Summary:")
            print(self.get_document_summary())
        
        print("\nCommands: 'quit' to exit, 'reload' for new document, 'summary' for document summary")
        print("'debug on/off' to toggle debug mode\n")
        
        debug_mode = False
        
        while True:
            try:
                question = input("❓ Your question: ").strip()
                
                if question.lower() == 'quit':
                    print("Goodbye!")
                    break
                elif question.lower() == 'reload':
                    self.document_chunks = []
                    self.document_name = ""
                    self.original_text = ""
                    print("Ready for new document...")
                    while not self.document_chunks:
                        doc_path = input("\nEnter path to your document: ").strip()
                        if not doc_path:
                            break
                        self.load_document(doc_path)
                    continue
                elif question.lower() == 'summary':
                    print("\n📄 Document Summary:")
                    print(self.get_document_summary())
                    print("-" * 60)
                    continue
                elif question.lower().startswith('debug'):
                    if 'on' in question.lower():
                        debug_mode = True
                        print("Debug mode enabled")
                    else:
                        debug_mode = False
                        print("Debug mode disabled")
                    continue
                elif not question:
                    continue
                
                print("\n🤔 Analyzing document...")
                answer = self.answer_question(question, debug=debug_mode)
                print(f"\n💡 Answer:\n{answer}\n")
                print("-" * 60)
                
            except KeyboardInterrupt:
                print("\n\nGoodbye!")
                break
            except Exception as e:
                print(f"Error: {str(e)}")

# Convenience functions
def create_enhanced_chatbot(document_path=None, model="llama3.1"):
    """Create an enhanced chatbot instance"""
    bot = EnhancedDocumentChatbot(model=model)
    if document_path:
        bot.load_document(document_path)
    return bot

def quick_enhanced_answer(document_path, question, model="llama3.1", debug=False):
    """Quick function to get an enhanced answer from a document"""
    bot = create_enhanced_chatbot(document_path, model)
    return bot.answer_question(question, debug=debug)

def main():
    """Main function with enhanced options"""
    # Check if running in Jupyter/IPython environment
    try:
        import sys
        if 'ipykernel' in sys.modules or 'IPython' in sys.modules:
            bot = EnhancedDocumentChatbot()
            bot.chat_loop()
            return
    except:
        pass
    
    # Command line mode
    parser = argparse.ArgumentParser(description="Enhanced Document Q&A Chatbot with Llama 3.1")
    parser.add_argument("--document", "-d", help="Path to document file")
    parser.add_argument("--question", "-q", help="Single question mode")
    parser.add_argument("--model", "-m", default="llama3.1", help="Ollama model to use")
    parser.add_argument("--debug", action="store_true", help="Enable debug mode")
    
    args, unknown = parser.parse_known_args()
    
    bot = EnhancedDocumentChatbot(model=args.model)
    
    if args.document:
        if bot.load_document(args.document):
            if args.question:
                answer = bot.answer_question(args.question, debug=args.debug)
                print(f"Question: {args.question}")
                print(f"Answer: {answer}")
            else:
                bot.chat_loop()
    else:
        bot.chat_loop()

if __name__ == "__main__":
    main()


🚀 Enhanced Document Q&A Chatbot with Llama 3.1
✓ Connected to Ollama (Model: llama3.1)
Loading document: G:\Campus Placements Assignments and Applications\Bajaj Technologies Assignment\D_Vamsidhar.pdf
✓ Document loaded successfully!
  - File: D_Vamsidhar.pdf
  - Text length: 5028 characters
  - Smart chunks created: 8
  - Average chunk size: 628 characters

Document 'D_Vamsidhar.pdf' is ready!

📄 Document Summary:
Error communicating with Ollama: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=45)

Commands: 'quit' to exit, 'reload' for new document, 'summary' for document summary
'debug on/off' to toggle debug mode


🤔 Analyzing document...

💡 Answer:
The document does not contain information about the number of internships Vamsidhar has done. However, it mentions that he is currently seeking an internship opportunity and lists one previous experience as a Geospatial AI Intern at HERE Technologies India Pvt. Ltd., which indicates that this may be his f

In [5]:
#!/usr/bin/env python3
"""
Enhanced Document-based Q&A Chatbot using Llama 3.1 via Ollama
Improved accuracy with semantic search, better chunking, and enhanced prompting
"""

import os
import requests
import json
import re
from pathlib import Path
import argparse
from typing import List, Dict, Optional, Tuple
from collections import Counter
import math

# Document processing imports
try:
    import PyPDF2
    from docx import Document
    PDF_SUPPORT = True
    DOCX_SUPPORT = True
except ImportError:
    PDF_SUPPORT = False
    DOCX_SUPPORT = False
    print("Install PyPDF2 and python-docx for full document support:")
    print("pip install PyPDF2 python-docx")

# Optional: Enhanced text processing
try:
    import nltk
    from nltk.tokenize import sent_tokenize, word_tokenize
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer
    NLTK_SUPPORT = True
    
    # Download required NLTK data
    try:
        nltk.data.find('tokenizers/punkt')
        nltk.data.find('corpora/stopwords')
    except LookupError:
        print("Downloading NLTK data...")
        nltk.download('punkt', quiet=True)
        nltk.download('stopwords', quiet=True)
        
except ImportError:
    NLTK_SUPPORT = False
    print("For better text processing, install NLTK: pip install nltk")

class EnhancedDocumentProcessor:
    """Enhanced document processing with better chunking and preprocessing"""
    
    def __init__(self):
        if NLTK_SUPPORT:
            self.stemmer = PorterStemmer()
            self.stop_words = set(stopwords.words('english'))
        else:
            self.stemmer = None
            self.stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should'}
    
    def load_document(self, file_path: str) -> str:
        """Load and extract text from various document formats"""
        path = Path(file_path)
        
        if not path.exists():
            raise FileNotFoundError(f"Document not found: {file_path}")
        
        if path.suffix.lower() == '.txt':
            return self._load_txt(file_path)
        elif path.suffix.lower() == '.pdf' and PDF_SUPPORT:
            return self._load_pdf(file_path)
        elif path.suffix.lower() == '.docx' and DOCX_SUPPORT:
            return self._load_docx(file_path)
        else:
            raise ValueError(f"Unsupported file format: {path.suffix}")
    
    def _load_txt(self, file_path: str) -> str:
        """Load text file"""
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    
    def _load_pdf(self, file_path: str) -> str:
        """Extract text from PDF"""
        text = ""
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
        return text
    
    def _load_docx(self, file_path: str) -> str:
        """Extract text from DOCX"""
        doc = Document(file_path)
        text = ""
        for paragraph in doc.paragraphs:
            text += paragraph.text + "\n"
        return text
    
    def preprocess_text(self, text: str) -> str:
        """Clean and preprocess text"""
        # Remove extra whitespace and normalize
        text = re.sub(r'\s+', ' ', text.strip())
        
        # Remove page numbers and headers/footers (simple heuristic)
        lines = text.split('\n')
        cleaned_lines = []
        
        for line in lines:
            line = line.strip()
            # Skip very short lines that might be page numbers or headers
            if len(line) < 10 and line.isdigit():
                continue
            # Skip lines with only special characters
            if re.match(r'^[^\w]*$', line):
                continue
            cleaned_lines.append(line)
        
        return '\n'.join(cleaned_lines)
    
    def smart_chunk_text(self, text: str, chunk_size: int = 800, overlap: int = 150) -> List[Dict]:
        """Enhanced chunking with semantic boundaries and metadata"""
        text = self.preprocess_text(text)
        
        if len(text) <= chunk_size:
            return [{"text": text, "start": 0, "end": len(text), "chunk_id": 0}]
        
        chunks = []
        chunk_id = 0
        
        # Try to use NLTK for sentence tokenization
        if NLTK_SUPPORT:
            sentences = sent_tokenize(text)
        else:
            # Fallback: simple sentence splitting
            sentences = re.split(r'[.!?]+', text)
            sentences = [s.strip() for s in sentences if s.strip()]
        
        current_chunk = ""
        current_start = 0
        
        for i, sentence in enumerate(sentences):
            # Calculate potential new chunk size
            potential_chunk = current_chunk + " " + sentence if current_chunk else sentence
            
            if len(potential_chunk) <= chunk_size:
                current_chunk = potential_chunk
            else:
                # Save current chunk if it has content
                if current_chunk:
                    chunk_end = current_start + len(current_chunk)
                    chunks.append({
                        "text": current_chunk.strip(),
                        "start": current_start,
                        "end": chunk_end,
                        "chunk_id": chunk_id,
                        "sentence_count": len(re.split(r'[.!?]+', current_chunk))
                    })
                    chunk_id += 1
                
                # Start new chunk with overlap
                if len(sentence) > chunk_size:
                    # Handle very long sentences
                    words = sentence.split()
                    for j in range(0, len(words), chunk_size // 10):
                        word_chunk = " ".join(words[j:j + chunk_size // 10])
                        chunks.append({
                            "text": word_chunk,
                            "start": text.find(word_chunk),
                            "end": text.find(word_chunk) + len(word_chunk),
                            "chunk_id": chunk_id,
                            "sentence_count": 1
                        })
                        chunk_id += 1
                else:
                    current_chunk = sentence
                    current_start = text.find(sentence, current_start)
        
        # Add final chunk
        if current_chunk:
            chunk_end = current_start + len(current_chunk)
            chunks.append({
                "text": current_chunk.strip(),
                "start": current_start,
                "end": chunk_end,
                "chunk_id": chunk_id,
                "sentence_count": len(re.split(r'[.!?]+', current_chunk))
            })
        
        return chunks

class SemanticSearcher:
    """Enhanced search using TF-IDF and semantic similarity"""
    
    def __init__(self):
        if NLTK_SUPPORT:
            self.stemmer = PorterStemmer()
            self.stop_words = set(stopwords.words('english'))
        else:
            self.stemmer = None
            self.stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should'}
    
    def preprocess_query(self, text: str) -> List[str]:
        """Preprocess text for semantic search"""
        # Convert to lowercase and tokenize
        if NLTK_SUPPORT:
            words = word_tokenize(text.lower())
        else:
            words = re.findall(r'\b\w+\b', text.lower())
        
        # Remove stopwords and stem
        processed_words = []
        for word in words:
            if word not in self.stop_words and len(word) > 2:
                if self.stemmer:
                    word = self.stemmer.stem(word)
                processed_words.append(word)
        
        return processed_words
    
    def calculate_tfidf_scores(self, query_words: List[str], chunks: List[Dict]) -> List[Tuple[float, Dict]]:
        """Calculate TF-IDF scores for relevance ranking"""
        # Calculate term frequencies in chunks
        chunk_word_counts = []
        all_words = set()
        
        for chunk in chunks:
            words = self.preprocess_query(chunk["text"])
            word_count = Counter(words)
            chunk_word_counts.append(word_count)
            all_words.update(words)
        
        # Calculate IDF (Inverse Document Frequency)
        idf = {}
        total_chunks = len(chunks)
        
        for word in all_words:
            chunks_with_word = sum(1 for wc in chunk_word_counts if word in wc)
            idf[word] = math.log(total_chunks / (chunks_with_word + 1))
        
        # Calculate TF-IDF scores for each chunk
        scores = []
        for i, chunk in enumerate(chunks):
            score = 0
            word_count = chunk_word_counts[i]
            total_words = sum(word_count.values())
            
            for query_word in query_words:
                if query_word in word_count:
                    tf = word_count[query_word] / total_words
                    score += tf * idf.get(query_word, 0)
            
            # Boost score based on chunk metadata
            if chunk.get("sentence_count", 1) > 3:  # Prefer chunks with more sentences
                score *= 1.1
            
            scores.append((score, chunk))
        
        return scores
    
    def find_relevant_chunks(self, question: str, chunks: List[Dict], max_chunks: int = 3) -> List[str]:
        """Find most relevant chunks using enhanced semantic search"""
        if not chunks:
            return []
        
        query_words = self.preprocess_query(question)
        if not query_words:
            return [chunk["text"] for chunk in chunks[:max_chunks]]
        
        # Calculate TF-IDF scores
        scored_chunks = self.calculate_tfidf_scores(query_words, chunks)
        
        # Sort by relevance score
        scored_chunks.sort(reverse=True, key=lambda x: x[0])
        
        # Filter out chunks with zero relevance
        relevant_chunks = [chunk for score, chunk in scored_chunks if score > 0]
        
        # If no relevant chunks found, fall back to first few chunks
        if not relevant_chunks:
            relevant_chunks = chunks[:max_chunks]
        else:
            relevant_chunks = relevant_chunks[:max_chunks]
        
        return [chunk["text"] if isinstance(chunk, dict) else chunk for chunk in relevant_chunks]

class EnhancedOllamaClient:
    """Enhanced Ollama client with improved prompting"""
    
    def __init__(self, base_url: str = "http://localhost:11434", model: str = "llama3.1"):
        self.base_url = base_url
        self.model = model
    
    def is_available(self) -> bool:
        """Check if Ollama is running"""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=5)
            return response.status_code == 200
        except:
            return False
    
    def check_model(self) -> bool:
        """Check if the specified model is available"""
        try:
            response = requests.get(f"{self.base_url}/api/tags")
            models = response.json().get('models', [])
            return any(self.model in model['name'] for model in models)
        except:
            return False
    
    def generate_response(self, prompt: str, context: str = "", question_type: str = "general") -> str:
        """Generate response with enhanced prompting"""
        full_prompt = self._build_enhanced_prompt(prompt, context, question_type)
        
        # Adjust parameters based on question type
        temperature = 0.3 if question_type == "factual" else 0.7
        
        payload = {
            "model": self.model,
            "prompt": full_prompt,
            "stream": False,
            "options": {
                "temperature": temperature,
                "top_p": 0.9,
                "max_tokens": 800,
                "stop": ["Human:", "Assistant:", "Context:"]
            }
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/api/generate", 
                json=payload,
                timeout=45
            )
            response.raise_for_status()
            return response.json()['response'].strip()
        except requests.RequestException as e:
            return f"Error communicating with Ollama: {str(e)}"
    
    def _classify_question(self, question: str) -> str:
        """Classify question type for better prompting"""
        question_lower = question.lower()
        
        if any(word in question_lower for word in ['what', 'who', 'when', 'where', 'which', 'how many']):
            return "factual"
        elif any(word in question_lower for word in ['why', 'how', 'explain', 'describe']):
            return "explanatory"
        elif any(word in question_lower for word in ['should', 'recommend', 'suggest', 'best']):
            return "advisory"
        else:
            return "general"
    
    def _build_enhanced_prompt(self, question: str, context: str, question_type: str = None) -> str:
        """Build enhanced prompt with better instructions"""
        if not question_type:
            question_type = self._classify_question(question)
        
        if context:
            return f"""You are an expert document analyst. Your task is to answer questions accurately based ONLY on the provided document context.

INSTRUCTIONS:
1. Read the context carefully and thoroughly
2. Answer based ONLY on information explicitly stated in the context
3. If the information is not in the context, clearly state "The document does not contain information about [specific aspect]"
4. Be precise and cite specific details from the context
5. If the context provides partial information, acknowledge what is known and what is missing
6. For {question_type} questions, focus on providing {self._get_question_guidance(question_type)}

DOCUMENT CONTEXT:
{context}

QUESTION: {question}

ANALYSIS AND ANSWER:
Based on the provided document context above, here is my response:"""
        else:
            return f"""I need document context to answer your question: {question}
Please provide relevant document content for me to analyze."""
    
    def _get_question_guidance(self, question_type: str) -> str:
        """Get specific guidance based on question type"""
        guidance = {
            "factual": "specific facts, numbers, dates, and concrete details",
            "explanatory": "clear explanations with supporting details from the document", 
            "advisory": "recommendations based on the information provided in the document",
            "general": "comprehensive information from the relevant sections"
        }
        return guidance.get(question_type, "accurate information from the document")

class EnhancedDocumentChatbot:
    """Enhanced chatbot with improved accuracy"""
    
    def __init__(self, model: str = "llama3.1"):
        self.processor = EnhancedDocumentProcessor()
        self.searcher = SemanticSearcher()
        self.ollama = EnhancedOllamaClient(model=model)
        self.document_chunks = []
        self.document_name = ""
        self.original_text = ""
    
    def load_document(self, file_path: str) -> bool:
        """Load and process document with enhanced processing"""
        print(f"Loading document: {file_path}")
        
        try:
            self.original_text = self.processor.load_document(file_path)
            self.document_chunks = self.processor.smart_chunk_text(self.original_text)
            self.document_name = Path(file_path).name
            
            print(f"✓ Document loaded successfully!")
            print(f"  - File: {self.document_name}")
            print(f"  - Text length: {len(self.original_text)} characters")
            print(f"  - Smart chunks created: {len(self.document_chunks)}")
            
            # Show sample chunk info
            if self.document_chunks:
                avg_chunk_size = sum(len(chunk["text"]) for chunk in self.document_chunks) / len(self.document_chunks)
                print(f"  - Average chunk size: {avg_chunk_size:.0f} characters")
            
        except Exception as e:
            print(f"✗ Error loading document: {str(e)}")
            return False
        
        return True
    
    def answer_question(self, question: str, max_chunks: int = 3, debug: bool = False) -> str:
        """Answer question with enhanced accuracy"""
        if not self.document_chunks:
            return "No document loaded. Please upload a document first."
        
        # Find relevant context using enhanced search
        relevant_chunks = self.searcher.find_relevant_chunks(question, self.document_chunks, max_chunks)
        
        if debug:
            print(f"Debug: Found {len(relevant_chunks)} relevant chunks")
            for i, chunk in enumerate(relevant_chunks):
                print(f"Chunk {i+1}: {chunk[:100]}...")
        
        context = "\n\n".join(relevant_chunks)
        
        # Classify question type for better prompting
        question_type = self.ollama._classify_question(question)
        
        if debug:
            print(f"Debug: Question classified as '{question_type}'")
        
        # Generate answer with enhanced prompting
        answer = self.ollama.generate_response(question, context, question_type)
        return answer
    
    def get_document_summary(self) -> str:
        """Generate a summary of the loaded document"""
        if not self.document_chunks:
            return "No document loaded."
        
        # Use first few chunks for summary
        summary_context = "\n".join([chunk["text"] for chunk in self.document_chunks[:3]])
        summary = self.ollama.generate_response(
            "Provide a brief summary of this document, highlighting the main topics and key points.",
            summary_context,
            "explanatory"
        )
        return summary
    
    def chat_loop(self):
        """Enhanced interactive chat loop"""
        print("\n" + "="*70)
        print("🚀 Enhanced Document Q&A Chatbot with Llama 3.1")
        print("="*70)
        
        # Check Ollama availability
        if not self.ollama.is_available():
            print("✗ Ollama is not running. Please start Ollama first:")
            print("  ollama serve")
            return
        
        if not self.ollama.check_model():
            print(f"✗ Model '{self.ollama.model}' not found. Please pull it first:")
            print(f"  ollama pull {self.ollama.model}")
            return
        
        print(f"✓ Connected to Ollama (Model: {self.ollama.model})")
        
        # Load document
        while not self.document_chunks:
            doc_path = input("\nEnter path to your document (PDF/TXT/DOCX): ").strip()
            if not doc_path:
                print("Goodbye!")
                return
            
            if not self.load_document(doc_path):
                continue
        
        # Offer document summary
        print(f"\nDocument '{self.document_name}' is ready!")
        if input("Would you like a document summary? (y/n): ").lower().startswith('y'):
            print("\n📄 Document Summary:")
            print(self.get_document_summary())
        
        print("\nCommands: 'quit' to exit, 'reload' for new document, 'summary' for document summary")
        print("'debug on/off' to toggle debug mode\n")
        
        debug_mode = False
        
        while True:
            try:
                question = input("❓ Your question: ").strip()
                
                if question.lower() == 'quit':
                    print("Goodbye!")
                    break
                elif question.lower() == 'reload':
                    self.document_chunks = []
                    self.document_name = ""
                    self.original_text = ""
                    print("Ready for new document...")
                    while not self.document_chunks:
                        doc_path = input("\nEnter path to your document: ").strip()
                        if not doc_path:
                            break
                        self.load_document(doc_path)
                    continue
                elif question.lower() == 'summary':
                    print("\n📄 Document Summary:")
                    print(self.get_document_summary())
                    print("-" * 60)
                    continue
                elif question.lower().startswith('debug'):
                    if 'on' in question.lower():
                        debug_mode = True
                        print("Debug mode enabled")
                    else:
                        debug_mode = False
                        print("Debug mode disabled")
                    continue
                elif not question:
                    continue
                
                print("\n🤔 Analyzing document...")
                answer = self.answer_question(question, debug=debug_mode)
                print(f"\n💡 Answer:\n{answer}\n")
                print("-" * 60)
                
            except KeyboardInterrupt:
                print("\n\nGoodbye!")
                break
            except Exception as e:
                print(f"Error: {str(e)}")

# Convenience functions
def create_enhanced_chatbot(document_path=None, model="llama3.1"):
    """Create an enhanced chatbot instance"""
    bot = EnhancedDocumentChatbot(model=model)
    if document_path:
        bot.load_document(document_path)
    return bot

def quick_enhanced_answer(document_path, question, model="llama3.1", debug=False):
    """Quick function to get an enhanced answer from a document"""
    bot = create_enhanced_chatbot(document_path, model)
    return bot.answer_question(question, debug=debug)

def main():
    """Main function with enhanced options"""
    # Check if running in Jupyter/IPython environment
    try:
        import sys
        if 'ipykernel' in sys.modules or 'IPython' in sys.modules:
            bot = EnhancedDocumentChatbot()
            bot.chat_loop()
            return
    except:
        pass
    
    # Command line mode
    parser = argparse.ArgumentParser(description="Enhanced Document Q&A Chatbot with Llama 3.1")
    parser.add_argument("--document", "-d", help="Path to document file")
    parser.add_argument("--question", "-q", help="Single question mode")
    parser.add_argument("--model", "-m", default="llama3.1", help="Ollama model to use")
    parser.add_argument("--debug", action="store_true", help="Enable debug mode")
    
    args, unknown = parser.parse_known_args()
    
    bot = EnhancedDocumentChatbot(model=args.model)
    
    if args.document:
        if bot.load_document(args.document):
            if args.question:
                answer = bot.answer_question(args.question, debug=args.debug)
                print(f"Question: {args.question}")
                print(f"Answer: {answer}")
            else:
                bot.chat_loop()
    else:
        bot.chat_loop()

if __name__ == "__main__":
    main()

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.

🚀 Enhanced Document Q&A Chatbot with Llama 3.1
✓ Connected to Ollama (Model: llama3.1)
Loading document: G:\Campus Placements Assignments and Applications\Bajaj Technologies Assignment\Bajaj Finserv Investor Presentation - FY2025-26 - Q1.pdf
✓ Document loaded successfully!
  - File: Bajaj Finserv Investor Presentation - FY2025-26 - Q1.pdf
  - Text length: 104116 characters
  - Smart chunks created: 246
  - Average chunk size: 499 characters

Document 'Bajaj Finserv Investor Presentation - FY2025-26 - Q1.pdf' is ready!

Commands: 'quit' to exit, 'reload' for new document, 'summary' for document summary
'debug on/off' to toggle debug mode


🤔 Analyzing document...

💡 Answer:
The document does not contain information about the gross NPAs (Non-Performing Assets) for Bajaj Finance. The document provides an overview of Bajaj Finserv Limited and its su